# Load CUAD dataset

Loads the official CUAD train/test JSON files and flattens them into a clause-level DataFrame
(one row per annotated span) with: contract ID, category, clause text, and span offsets.

Data files are in `data/cuad/` (see `data/cuad/README.md`). We use the official
`train_separate_questions.json` (408 contracts) and `test.json` (102 contracts) splits as-is.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/cuad")
TRAIN_PATH = DATA_DIR / "train_separate_questions.json"
TEST_PATH = DATA_DIR / "test.json"

## Flatten the JSON

Each file is SQuAD-style: `data[] -> paragraphs[] -> qas[] -> answers[]`. Every contract has one paragraph
whose `context` is the full contract text, and one question per category.

Note: in `train_separate_questions.json` each answer span is split out into its own question, so the
question ids end in `_0`, `_1`, ... (e.g. `...__Parties_3`). We strip that suffix to recover the category name.
`test.json` has one question per category with multiple answers instead.

In [2]:
def category_from_id(qid: str) -> str:
    """'<contract title>__Parties_3' -> 'Parties'"""
    return re.sub(r"_\d+$", "", qid.split("__")[-1])


def load_cuad(path):
    """Return (clauses_df, contracts_df).

    clauses_df:   one row per annotated span
    contracts_df: one row per contract with the full text
    """
    with open(path) as f:
        data = json.load(f)["data"]

    clause_rows, contract_rows = [], []
    for contract in data:
        contract_id = contract["title"]
        for para in contract["paragraphs"]:
            contract_rows.append({"contract_id": contract_id, "text": para["context"]})
            for qa in para["qas"]:
                category = category_from_id(qa["id"])
                for ans in qa["answers"]:
                    start = ans["answer_start"]
                    clause_rows.append({
                        "contract_id": contract_id,
                        "category": category,
                        "clause_text": ans["text"],
                        "start": start,
                        "end": start + len(ans["text"]),
                    })

    return pd.DataFrame(clause_rows), pd.DataFrame(contract_rows)

In [3]:
train_clauses, train_contracts = load_cuad(TRAIN_PATH)
test_clauses, test_contracts = load_cuad(TEST_PATH)

print(f"train: {len(train_contracts)} contracts, {len(train_clauses)} clause spans")
print(f"test:  {len(test_contracts)} contracts, {len(test_clauses)} clause spans")

train: 408 contracts, 11180 clause spans
test:  102 contracts, 2643 clause spans


In [4]:
train_clauses.head(10)

,contract_id,category,clause_text,start,end
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,DISTRIBUTOR AGREEMENT,44,65
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Distributor,244,255
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Electric City of Illinois L.L.C.,49574,49606
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Electric City of Illinois LLC,212,241
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Company,197,204
5,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Electric City Corp.,148,167
6,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,"7th day of September, 1999.",263,290
7,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,Unless earlier terminated otherwise prov...,31058,31270
8,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,The term of this Agreement shall be ten (10)...,5268,5541
9,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,The term of this Agreement shall be ten (10)...,5268,5541


In [5]:
train_contracts.head()

,contract_id,text
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...


## Sanity checks

- 41 categories in train. Test only has 40: `Price Restrictions` has no positive spans in any test contract
  (it is a rare category, ~4% of train contracts).
- Span offsets point at the clause text in the contract

In [6]:
train_cats = set(train_clauses["category"])
test_cats = set(test_clauses["category"])
print("train categories:", len(train_cats))
print("test categories: ", len(test_cats))
print("in train but not test:", train_cats - test_cats)
assert len(train_cats) == 41 and test_cats <= train_cats

train categories: 41
test categories:  40
in train but not test: {'Price Restrictions'}


In [7]:
def check_offsets(clauses, contracts):
    text_by_id = contracts.set_index("contract_id")["text"]
    sliced = [
        text_by_id[row.contract_id][row.start:row.end]
        for row in clauses.itertuples()
    ]
    return (pd.Series(sliced) == clauses["clause_text"].values).mean()

print("train offset match rate:", check_offsets(train_clauses, train_contracts))
print("test offset match rate: ", check_offsets(test_clauses, test_contracts))

train offset match rate: 1.0
test offset match rate:  1.0


## Contract-level label matrix

A multi-hot `contracts x 41 categories` matrix (1 if the contract has at least one span for that category).
Useful for the contract-level split (#8) and the EDA on class imbalance (#9).

In [8]:
def label_matrix(clauses, contracts):
    mat = pd.crosstab(clauses["contract_id"], clauses["category"]).clip(upper=1)
    # keep every contract, even ones with no spans, and a fixed column order
    return mat.reindex(index=contracts["contract_id"], columns=sorted(mat.columns), fill_value=0)

train_labels = label_matrix(train_clauses, train_contracts)
test_labels = label_matrix(test_clauses, test_contracts)

print(train_labels.shape, test_labels.shape)
train_labels.sum().sort_values(ascending=False)

(408, 41) (102, 40)


category
Document Name                         408
Parties                               407
Agreement Date                        377
Governing Law                         354
Expiration Date                       335
Effective Date                        320
Anti-Assignment                       302
Cap On Liability                      231
License Grant                         205
Audit Rights                          176
Termination For Convenience           154
Post-Termination Services             153
Renewal Term                          150
Exclusivity                           147
Insurance                             134
Minimum Commitment                    133
Revenue/Profit Sharing                131
Non-Transferable License              116
Ip Ownership Assignment               101
Uncapped Liability                     98
Non-Compete                            96
Change Of Control                      95
Notice Period To Terminate Renewal     95
Covenant Not To Sue      